In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [15]:
data = pd.read_csv('../dataset/cleaned_dataset.csv', index_col = 0)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [16]:
## Dividing into independent and dependent features
X = data.drop('EstimatedSalary', axis=1)
y = data['EstimatedSalary']

In [17]:
## Split the data in training and testing
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [18]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

## ANN Implementation

In [19]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [20]:
# Build the model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)  # Output layer for regression
])

## compile the model
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [22]:
# Set up Early Stopping
from tensorflow.keras.callbacks import EarlyStopping

early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [23]:
# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[early_stopping_callback]
)

Epoch 1/100


250/250 [==============================] - 3s 5ms/step - loss: 100371.0234 - mae: 100371.0234 - val_loss: 98486.2969 - val_mae: 98486.2969
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 99509.6719 - mae: 99509.6719 - val_loss: 96739.4688 - val_mae: 96739.4688
Epoch 3/100
250/250 [==============================] - 1s 4ms/step - loss: 96481.4844 - mae: 96481.4844 - val_loss: 92302.9453 - val_mae: 92302.9453
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 90595.1484 - mae: 90595.1484 - val_loss: 85065.6797 - val_mae: 85065.6797
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 82192.7578 - mae: 82192.7578 - val_loss: 75915.4609 - val_mae: 75915.4609
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 72625.6953 - mae: 72625.6953 - val_loss: 66797.9766 - val_mae: 66797.9766
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 63731.2070 - mae: 63731.207

In [24]:
## Evaluate model on the test data
test_loss,test_mae=model.evaluate(X_test,y_test)
print(f'Test MAE : {test_mae}')

63/63 [==============================] - 0s 2ms/step - loss: 50347.4531 - mae: 50347.4531
Test MAE : 50347.453125


In [25]:
model.save('../models/ann_regressor.h5')

g:\Deep Learning ANN\ANN\venv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
